In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import time

print("Current working directory:")
print(Path.cwd())

# 因为当前 notebook 在 MLF_coursework/Stage_4 里面，
# 而 stage4_model_panel.parquet 在上一层 MLF_coursework 根目录下，
# 所以这里用 ../
INPUT_PATH = Path("stage4_model_panel.parquet")
CONFIG_PATH = Path("stage4_feature_config.json")

# 清洗后的建模文件也保存到 MLF_coursework 根目录
OUTPUT_PATH = Path("stage4_model_data_clean.parquet")

# 诊断输出放到 MLF_coursework/stage4_diagnostics
REPORT_DIR = Path("stage4_diagnostics")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

REPORT_MISSING_BEFORE = REPORT_DIR / "stage4_missing_rate_before_cleaning.csv"
REPORT_MISSING_AFTER = REPORT_DIR / "stage4_missing_rate_after_cleaning.csv"
REPORT_WINSOR_BOUNDS = REPORT_DIR / "stage4_winsor_bounds.csv"
REPORT_IMPUTE_VALUES = REPORT_DIR / "stage4_impute_values.csv"
REPORT_CLEANING_SUMMARY = REPORT_DIR / "stage4_cleaning_summary.json"
REPORT_DESCRIBE_AFTER = REPORT_DIR / "stage4_feature_describe_after_cleaning.csv"

LOWER_Q = 0.01
UPPER_Q = 0.99

print("\nChecking files:")
print("INPUT_PATH exists:", INPUT_PATH.exists(), INPUT_PATH)
print("CONFIG_PATH exists:", CONFIG_PATH.exists(), CONFIG_PATH)

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Cannot find input file: {INPUT_PATH}")

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Cannot find config file: {CONFIG_PATH}")

print("\nPath setup complete.")

In [ ]:
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = json.load(f)

features = config["features"]
target = config["target"]

print("Target:", target)
print("Number of features:", len(features))

id_cols = [
    "date",
    "year",
    "ticker",
    "instrument_id",
    "is_trade_eligible",
    "eligibility_reason",
    "sample_split",
]

cols_to_read = id_cols + [target] + features

# 去重，保持顺序
cols_to_read = list(dict.fromkeys(cols_to_read))

print("\nColumns to read:", len(cols_to_read))
for c in cols_to_read:
    print(c)

In [ ]:
start = time.time()
print("Reading selected columns from stage4_model_panel.parquet ...", flush=True)

panel = pd.read_parquet(INPUT_PATH, columns=cols_to_read)

print("Read finished.")
print("Shape:", panel.shape)
print("Time used:", round(time.time() - start, 2), "seconds")

panel["date"] = pd.to_datetime(panel["date"])

panel.head()

In [ ]:
# 先把 target 里的 inf 替换成 NaN，避免 mask 出问题
panel[target] = panel[target].replace([np.inf, -np.inf], np.nan)

train_mask = (
    (panel["sample_split"] == "train") &
    (panel["is_trade_eligible"] == True) &
    (panel[target].notna())
)

eligible_mask = (
    (panel["is_trade_eligible"] == True) &
    (panel[target].notna())
)

print("Total rows:", len(panel))
print("Train eligible rows:", int(train_mask.sum()))
print("All eligible rows:", int(eligible_mask.sum()))

print("\nSample split counts:")
print(panel["sample_split"].value_counts(dropna=False))

print("\nTrade eligible counts:")
print(panel["is_trade_eligible"].value_counts(dropna=False))


In [ ]:
start = time.time()

print("Replacing inf and -inf with NaN in feature columns ...", flush=True)
panel[features] = panel[features].replace([np.inf, -np.inf], np.nan)

print("Checking missing rates before cleaning ...", flush=True)

missing_before = (
    panel.loc[eligible_mask, features + [target]]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

missing_before.columns = ["column", "missing_rate_before"]
missing_before.to_csv(REPORT_MISSING_BEFORE, index=False)

print("Time used:", round(time.time() - start, 2), "seconds")
missing_before.head(30)

In [ ]:
numeric_features = [
    c for c in features
    if pd.api.types.is_numeric_dtype(panel[c])
]

non_numeric_features = [
    c for c in features
    if c not in numeric_features
]

calendar_features = [
    c for c in ["month", "day_of_week", "is_month_end", "is_quarter_end"]
    if c in features
]

binary_features = []

for c in numeric_features:
    vals = panel.loc[train_mask, c].dropna().unique()
    if len(vals) <= 2:
        binary_features.append(c)

continuous_features = [
    c for c in numeric_features
    if c not in binary_features and c not in calendar_features
]

print("Numeric features:", len(numeric_features))
print("Non-numeric features:", non_numeric_features)
print("Binary features:", binary_features)
print("Calendar features:", calendar_features)
print("Continuous features for winsorization:", len(continuous_features))

continuous_features

In [ ]:
start = time.time()
print("Computing train-only median imputation values ...", flush=True)

# 只用 train eligible 样本计算 median
medians = panel.loc[train_mask, numeric_features].median()

# 如果某个变量 median 还是 NaN，用 0 兜底
medians = medians.fillna(0.0)

# 保存填充值
impute_df = medians.reset_index()
impute_df.columns = ["feature", "impute_value"]
impute_df.to_csv(REPORT_IMPUTE_VALUES, index=False)

# 用 train median 填补全样本缺失值
panel[numeric_features] = panel[numeric_features].fillna(medians)

# 如果有非数值特征，用 train mode 填补
for c in non_numeric_features:
    mode_value = panel.loc[train_mask, c].mode(dropna=True)
    fill_value = mode_value.iloc[0] if len(mode_value) > 0 else "missing"
    panel[c] = panel[c].fillna(fill_value)

print("Imputation finished.")
print("Time used:", round(time.time() - start, 2), "seconds")

impute_df.head(30)

In [ ]:
start = time.time()
print("Computing train-only winsorization bounds ...", flush=True)

# 一次性计算所有连续变量的 1% 和 99% 分位数
bounds = panel.loc[train_mask, continuous_features].quantile([LOWER_Q, UPPER_Q])

winsor_rows = []

for i, c in enumerate(continuous_features, start=1):
    print(f"Winsorizing {i}/{len(continuous_features)}: {c}", flush=True)

    lower = bounds.loc[LOWER_Q, c]
    upper = bounds.loc[UPPER_Q, c]

    if pd.isna(lower) or pd.isna(upper) or lower >= upper:
        print(f"  skipped {c}", flush=True)
        continue

    n_low = int((panel[c] < lower).sum())
    n_high = int((panel[c] > upper).sum())

    panel[c] = panel[c].clip(lower=lower, upper=upper)

    winsor_rows.append({
        "feature": c,
        "lower_q": LOWER_Q,
        "upper_q": UPPER_Q,
        "lower_bound": float(lower),
        "upper_bound": float(upper),
        "n_clipped_low": n_low,
        "n_clipped_high": n_high,
        "n_clipped_total": n_low + n_high,
    })

winsor_df = pd.DataFrame(winsor_rows)
winsor_df.to_csv(REPORT_WINSOR_BOUNDS, index=False)

print("Winsorization finished.")
print("Time used:", round(time.time() - start, 2), "seconds")

winsor_df.sort_values("n_clipped_total", ascending=False).head(30)

In [ ]:
start = time.time()
print("Creating winsorized target column ...", flush=True)

target_winsor = target + "_winsor"

lower_y = panel.loc[train_mask, target].quantile(LOWER_Q)
upper_y = panel.loc[train_mask, target].quantile(UPPER_Q)

panel[target_winsor] = panel[target].clip(lower=lower_y, upper=upper_y)

print("Created:", target_winsor)
print("Target lower bound:", lower_y)
print("Target upper bound:", upper_y)
print("Time used:", round(time.time() - start, 2), "seconds")

panel[[target, target_winsor]].describe()

In [ ]:
start = time.time()
print("Checking missing rates after cleaning ...", flush=True)

missing_after = (
    panel.loc[eligible_mask, features + [target]]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

missing_after.columns = ["column", "missing_rate_after"]
missing_after.to_csv(REPORT_MISSING_AFTER, index=False)

print("Time used:", round(time.time() - start, 2), "seconds")
missing_after.head(30)

In [ ]:
start = time.time()
print("Saving clean modelling file ...", flush=True)

panel.to_parquet(OUTPUT_PATH, index=False, compression="snappy")

print("Saved to:", OUTPUT_PATH.resolve())
print("Final shape:", panel.shape)
print("Time used:", round(time.time() - start, 2), "seconds")

In [ ]:
summary = {
    "input_file": str(INPUT_PATH),
    "output_file": str(OUTPUT_PATH),
    "n_rows": int(len(panel)),
    "n_columns": int(panel.shape[1]),
    "n_features": int(len(features)),
    "target": target,
    "winsorized_target_column": target_winsor,
    "train_eligible_rows_used_for_preprocessing": int(train_mask.sum()),
    "eligible_rows": int(eligible_mask.sum()),
    "lower_quantile": LOWER_Q,
    "upper_quantile": UPPER_Q,
    "n_numeric_features": int(len(numeric_features)),
    "n_continuous_features_winsorized": int(len(continuous_features)),
    "n_binary_features_not_winsorized": int(len(binary_features)),
    "n_calendar_features_not_winsorized": int(len(calendar_features)),
    "missing_report_before": str(REPORT_MISSING_BEFORE),
    "missing_report_after": str(REPORT_MISSING_AFTER),
    "impute_values_file": str(REPORT_IMPUTE_VALUES),
    "winsor_bounds_file": str(REPORT_WINSOR_BOUNDS),
}

with open(REPORT_CLEANING_SUMMARY, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Cleaning summary saved to:")
print(REPORT_CLEANING_SUMMARY.resolve())

summary


In [ ]:
clean_panel = pd.read_parquet(OUTPUT_PATH)

print("Clean panel shape:", clean_panel.shape)

print("\nMissing rates in features:")
print(clean_panel[features].isna().mean().sort_values(ascending=False).head(20))

print("\nColumns:")
print(clean_panel.columns.tolist())

clean_panel.head()

read clean data

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

# ============================================================
# 1. 设置文件路径
# ============================================================

CLEAN_DATA_PATH = Path("stage4_model_data_clean.parquet")
CONFIG_PATH = Path("stage4_feature_config.json")

print("Current working directory:")
print(Path.cwd())

print("\nChecking files:")
print("Clean data exists:", CLEAN_DATA_PATH.exists(), CLEAN_DATA_PATH)
print("Feature config exists:", CONFIG_PATH.exists(), CONFIG_PATH)

if not CLEAN_DATA_PATH.exists():
    raise FileNotFoundError(f"Cannot find clean data file: {CLEAN_DATA_PATH}")

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Cannot find feature config file: {CONFIG_PATH}")


# ============================================================
# 2. 读取清洗后的数据和 feature config
# ============================================================

panel = pd.read_parquet(CLEAN_DATA_PATH)

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = json.load(f)

features = config["features"]
target = config["target"]

target_winsor = target + "_winsor"

print("\nData loaded successfully.")
print("Panel shape:", panel.shape)
print("Target:", target)
print("Winsorized target exists:", target_winsor in panel.columns)
print("Number of model features:", len(features))


# ============================================================
# 3. 输出所有变量名
# ============================================================

print("\nAll columns in cleaned data:")
for i, col in enumerate(panel.columns, start=1):
    print(f"{i:02d}. {col}")


# ============================================================
# 4. 输出模型真正会用到的特征
# ============================================================

print("\nModel features from config:")
for i, col in enumerate(features, start=1):
    print(f"{i:02d}. {col}")


# ============================================================
# 5. 检查特征是否都存在
# ============================================================

missing_features = [c for c in features if c not in panel.columns]

if missing_features:
    print("\nMissing features:")
    print(missing_features)
else:
    print("\nAll model features exist in the cleaned data.")


# ============================================================
# 6. 按类型粗略分类变量
# ============================================================

id_cols = [
    "date", "year", "ticker", "instrument_id",
    "is_trade_eligible", "eligibility_reason", "sample_split"
]

target_cols = [target]
if target_winsor in panel.columns:
    target_cols.append(target_winsor)

return_features = [
    c for c in features
    if c.startswith("r_") or c.startswith("vol_") or c.startswith("abs_")
]

liquidity_size_features = [
    c for c in features
    if c in [
        "price_for_filter",
        "adv20",
        "ann_vol60",
        "year_start_market_cap",
        "market_cap_rank"
    ]
]

short_interest_features = [
    c for c in features
    if c in ["dsi", "dtcn", "ddtcn"]
]

calendar_features = [
    c for c in features
    if c in ["month", "day_of_week", "is_month_end", "is_quarter_end"]
]

print("\nFeature groups:")
print("ID / metadata columns:", [c for c in id_cols if c in panel.columns])
print("Target columns:", target_cols)
print("Return / volatility features:", return_features)
print("Liquidity / size features:", liquidity_size_features)
print("Short-interest features:", short_interest_features)
print("Calendar features:", calendar_features)


# ============================================================
# 7. 查看每个特征的数据类型、缺失率、唯一值数量
# ============================================================

feature_overview = pd.DataFrame({
    "column": features,
    "dtype": [str(panel[c].dtype) for c in features],
    "missing_rate": [panel[c].isna().mean() for c in features],
    "n_unique": [panel[c].nunique(dropna=True) for c in features],
})

feature_overview = feature_overview.sort_values("missing_rate", ascending=False)

print("\nFeature overview:")
display(feature_overview)


# ============================================================
# 8. 查看 target 和 sample split 情况
# ============================================================

print("\nSample split counts:")
display(panel["sample_split"].value_counts(dropna=False))

print("\nTrade eligible counts:")
display(panel["is_trade_eligible"].value_counts(dropna=False))

print("\nTarget summary:")
display(panel[target].describe())

if target_winsor in panel.columns:
    print("\nWinsorized target summary:")
    display(panel[target_winsor].describe())


# ============================================================
# 9. 只看真正用于建模的 eligible 样本
# ============================================================

model_df = panel[
    (panel["is_trade_eligible"] == True) &
    (panel[target].notna())
].copy()

print("\nModel sample shape:")
print(model_df.shape)

print("\nModel sample by split:")
display(
    model_df
    .groupby("sample_split")[target]
    .agg(
        rows="size",
        target_mean="mean",
        target_std="std",
        target_min="min",
        target_max="max"
    )
)


# ============================================================
# 10. 查看前几行关键内容
# ============================================================

preview_cols = [
    "date", "ticker", "instrument_id",
    "is_trade_eligible", "sample_split",
    target
]

if target_winsor in panel.columns:
    preview_cols.append(target_winsor)

preview_cols += features[:10]

preview_cols = [c for c in preview_cols if c in panel.columns]

print("\nPreview rows:")
display(panel[preview_cols].head(20))

增添liquidity / volatility / short-interest / calendar 特征